In [50]:
import MDAnalysis as mda
import pandas as pd
from biopandas.pdb import PandasPdb
import os
import glob
import re
import math
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

def pdb_to_dataframe(pdb_file):
    """
    Load a PDB file using MDAnalysis and convert key atom information to a pandas DataFrame.
    """
    u = mda.Universe(pdb_file)
    
    # Extract atom-related data: atom name, residue name, residue ID, and chain ID
    atom_data = {
        'Atom Name': u.atoms.names,
        'Residue Name': u.atoms.resnames,
        'Residue ID': u.atoms.resids,
        'Chain ID': u.atoms.segids,
        'X': u.atoms.positions[:, 0],
        'Y': u.atoms.positions[:, 1],
        'Z': u.atoms.positions[:, 2],
    }
    
    # Create a pandas DataFrame from the atom data
    df = pd.DataFrame(atom_data)
    
    return df

def grid_list(atom_df):
    return list(zip(atom_df['x_coord'], atom_df['y_coord'], atom_df['z_coord']))

def filtering_proteins(atom_df, grid_list, radius=5.0):
    import numpy as np

    # Adjust these column names if needed for your dataframe
    residue_cols = ['chain_id', 'residue_name', 'residue_number', 'insertion']
    residue_cols = [col for col in residue_cols if col in atom_df.columns]

    if not residue_cols:
        raise ValueError("Could not identify residue columns in atom_df.")

    atom_coords = atom_df[['x_coord', 'y_coord', 'z_coord']].values
    initially_filtered_atoms = set()

    # Step 1: find atoms within radius of any grid point
    for x, y, z in grid_list:
        distances_sq = (
            (atom_coords[:, 0] - x) ** 2 +
            (atom_coords[:, 1] - y) ** 2 +
            (atom_coords[:, 2] - z) ** 2
        )
        mask = distances_sq <= radius ** 2
        initially_filtered_atoms.update(atom_df.index[mask])

    print(f"Total atoms within {radius} Å cutoff: {len(initially_filtered_atoms)}")

    if not initially_filtered_atoms:
        return atom_df.loc[list(initially_filtered_atoms)]

    grouped = atom_df.groupby(residue_cols)

    # Step 2: residues passing the 50% rule
    residue_keep_set = set()

    for residue_key, residue_df in grouped:
        residue_atom_indices = set(residue_df.index)
        n_total = len(residue_atom_indices)
        n_filtered = len(residue_atom_indices & initially_filtered_atoms)

        if n_total == 0:
            continue

        fraction_present = n_filtered / n_total

        if fraction_present >= 0.5:
            residue_keep_set.add(residue_key)

    # Step 3: expand to all atoms in kept residues
    filtered_atoms = set()
    for residue_key, residue_df in grouped:
        if residue_key in residue_keep_set:
            filtered_atoms.update(residue_df.index)

    # Fallback: if nothing survives 50% rule, keep residues that had any atom in cutoff
    if len(filtered_atoms) == 0:
        print("No residues passed the 50% occupancy filter. Falling back to residues with at least one atom within cutoff.")

        fallback_residue_keep_set = set()

        for residue_key, residue_df in grouped:
            residue_atom_indices = set(residue_df.index)
            if len(residue_atom_indices & initially_filtered_atoms) > 0:
                fallback_residue_keep_set.add(residue_key)

        for residue_key, residue_df in grouped:
            if residue_key in fallback_residue_keep_set:
                filtered_atoms.update(residue_df.index)

    print(f"Total atoms after residue expansion + filtering: {len(filtered_atoms)}")
    return atom_df.loc[list(filtered_atoms)]


In [51]:
def get_positive_ligand_atoms(positive_file, protein_name):
    protein_pdb_df = PandasPdb().read_pdb(positive_file)
    protein_pdb_df.df.keys()
    protein = protein_pdb_df.df['ATOM']
    protein = protein[~protein['atom_name'].str.startswith('H')] # don't use hydrogen
    protein_coords = protein[['x_coord', 'y_coord', 'z_coord']].values
    protein_centroid = protein_coords.mean(axis=0)
    print(set(protein['chain_id']))
    print(positive_file)

    ligand_df = PandasPdb().read_pdb(positive_file)
    ligand_df.df.keys()
    ligand = ligand_df.df['HETATM']
    ligand = ligand[ligand['residue_name']=="CLR"]
    x = list(set(zip(ligand['residue_number'], ligand['chain_id'])))

    #get the most inward residue
    min_distance = float('inf')
    closest_clr = None

    all_ligands = []

    for residue_number, chain_id in x:
        clr_atoms = ligand[(ligand['residue_number'] == residue_number) & (ligand['chain_id'] == chain_id)]
        if clr_atoms.empty:
            continue

        clr_coords = clr_atoms[['x_coord', 'y_coord', 'z_coord']].values
        clr_centroid = clr_coords.mean(axis=0)
        
        distance = np.linalg.norm(protein_centroid - clr_centroid)
        
        if distance < min_distance:
            min_distance = distance
            closest_clr = (residue_number, chain_id)

        grid_list_ = grid_list(clr_atoms)

        all_ligands.append(filtering_proteins(protein, grid_list_))

    ligand_ = ligand[(ligand['residue_number'] == closest_clr[0]) & (ligand['chain_id'] == closest_clr[1])]
    grid_list_ = grid_list(ligand_)

    filtered_atoms = filtering_proteins(protein, grid_list_)

    # Save to pdb
    filtered_pdb = PandasPdb()
    filtered_pdb.df['ATOM'] = filtered_atoms
    filtered_pdb_path = f"filtered-rdkit-distinct-5A/positive/{protein_name}-filtered.pdb"
    os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
    filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)

    return protein, all_ligands


In [52]:
def check_if_unlabeled_is_positive(positive_df, unlabeled_df):
    # Create a unique key for each atom based on identifying features
    positive_df['atom_key'] = (
        positive_df['atom_name'].str.strip() + '_' +
        positive_df['residue_name'].str.strip() + '_' +
        positive_df['residue_number'].astype(str) + '_' +
        positive_df['chain_id'].fillna('')
    )

    unlabeled_df['atom_key'] = (
        unlabeled_df['atom_name'].str.strip() + '_' +
        unlabeled_df['residue_name'].str.strip() + '_' +
        unlabeled_df['residue_number'].astype(str) + '_' +
        unlabeled_df['chain_id'].fillna('')
    )

    keys1 = set(positive_df['atom_key'])
    keys2 = set(unlabeled_df['atom_key'])

    common_atoms = keys1 & keys2
    total_atoms = max(len(keys1), len(keys2))

    if total_atoms == 0:
        print("Zero total atoms")
        return False

    similarity = len(common_atoms) / total_atoms
    print(similarity)
    return similarity >= 0.2

In [53]:
def get_protein_name(filename):
    basename = os.path.basename(filename)  # Get file name without path
    match = re.match(r'([a-zA-Z0-9]{4})', basename)  # Match the first 4-character PDB ID
    if match:
        return match.group(1).upper()
    else:
        return None
def get_mode_index(filename):
    basename = os.path.basename(filename)
    match = re.search(r'mode_(\d+)', basename)
    if match:
        return int(match.group(1))
    else:
        return None  # or raise ValueError("No mode index found.")

def natural_sort_key(s):
    """Function to sort strings in a natural alphanumeric order."""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]


In [54]:
# positive_files = glob.glob("../GNN/CLR-PDB/*.pdb")
# positive_files = sorted(positive_files, key=natural_sort_key)

# unlabeled_files = glob.glob("../GNN/CLR-Unlabeled-Distinct/*.pdb")
# unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# positive_index = 0
# protein, all_lig_filtered = get_positive_ligand_atoms(positive_files[positive_index], get_protein_name(positive_files[positive_index]))

# for unlabeled_file in unlabeled_files:
#     positive_name = get_protein_name(positive_files[positive_index])
#     unlabeled_name = get_protein_name(unlabeled_file)

#     fragment_index = get_mode_index(unlabeled_file)

#     if positive_name != unlabeled_name:
#         positive_index += 1
#         positive_name = get_protein_name(positive_files[positive_index])

#         if positive_name != unlabeled_name:
#             raise Exception("Proteins Not Matching Up!!!")
        
#         protein, all_lig_filtered = get_positive_ligand_atoms(positive_files[positive_index], positive_name)

#     fragment_df = PandasPdb().read_pdb(unlabeled_file)
#     fragment_df.df.keys()
#     fragment = fragment_df.df['HETATM']

#     grid_list_ = grid_list(fragment)

#     filtered_atoms = filtering_proteins(protein, grid_list_)
    
#     if not filtered_atoms.empty:
#         for lig in all_lig_filtered:
#             is_positive = check_if_unlabeled_is_positive(lig, filtered_atoms)

#             if is_positive:
#                 break

#         # Save to pdb
#         filtered_pdb = PandasPdb()
#         filtered_pdb.df['ATOM'] = filtered_atoms

#         if is_positive:
#             filtered_pdb_path = f"filtered-rdkit-distinct-5A/unlabeled/{unlabeled_name}-f{fragment_index}-positive.pdb"
#         else:
#             filtered_pdb_path = f"filtered-rdkit-distinct-5A/unlabeled/{unlabeled_name}-f{fragment_index}.pdb"
#         os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
#         filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)
    
#     fragment_index += 1

In [55]:
import numpy as np

import numpy as np
from rdkit import Chem

ATOM_SUBTYPES = [
    # Carbon (17)
    'C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3',
    'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3',

    # Oxygen (8)
    'O', 'OH', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1',

    # Nitrogen (9)
    'N', 'NE', 'NE1', 'NE2', 'ND1', 'ND2', 'NZ', 'NH1', 'NH2',

    # Sulfur (2)
    'SD', 'SG',

    # Unknown (1)
    'UNKNOWN'
]

ATOM_TO_INDEX = {atom: idx for idx, atom in enumerate(ATOM_SUBTYPES)}

N_ATOM_SUBTYPE = len(ATOM_SUBTYPES)   # 37
N_HYBRID = 3                          # SP, SP2, SP3
N_MISC = 3                            # charge, ring, aromatic
N_RESIDUE_GROUPS = 9                  # ASP/GLU, LYS/ARG, HIS, CYS, ASN/GLN/SER/THR, GLY, PRO, PHE/TYR/TRP, ALA/ILE/LEU/MET/VAL
ENCODING_SIZE = N_ATOM_SUBTYPE + N_HYBRID + N_MISC + N_RESIDUE_GROUPS  # 52


def one_hot_encoding(pdb_df, dtype=np.int8):
    """
    Creates a (num_atoms, 37) one-hot matrix using pdb_df['Atom Name'].
    """
    num_rows = len(pdb_df)
    one_hot_matrix = np.zeros((num_rows, N_ATOM_SUBTYPE), dtype=dtype)

    for i, atom_name in enumerate(pdb_df['Atom Name']):
        atom_name = str(atom_name).strip()
        idx = ATOM_TO_INDEX.get(atom_name, ATOM_TO_INDEX['UNKNOWN'])
        one_hot_matrix[i, idx] = 1

        if atom_name not in ATOM_TO_INDEX:
            print(f"{atom_name} went to UNKNOWN column")

    return one_hot_matrix


def rdkit_coords_and_encoding(pdb_path, pdb_df, output_dtype=np.float32):
    """
    Returns:
        coords: (N, 3) float32
        encoded_atoms: (N, 52) float32 by default

    First 37 columns = atom subtype one-hot from pdb_df['Atom Name']
    Remaining columns = RDKit-derived features
    """
    mol = Chem.MolFromPDBFile(pdb_path, sanitize=True)
    if mol is None:
        print(f"[SKIP] RDKit could not parse: {pdb_path}")
        return None, None

    conf = mol.GetConformer()
    num_atoms = mol.GetNumAtoms()

    # Important safety check:
    if len(pdb_df) != num_atoms:
        print(f"[WARNING] Atom count mismatch for {pdb_path}: "
              f"pdb_df has {len(pdb_df)} rows, RDKit mol has {num_atoms} atoms")
        return None, None

    # Build 37-column atom subtype one-hot from PDB atom names
    subtype_one_hot = one_hot_encoding(pdb_df, dtype=np.int8)

    coords = np.zeros((num_atoms, 3), dtype=np.float32)
    encoded_atoms = np.zeros((num_atoms, ENCODING_SIZE), dtype=np.int8)

    # Put subtype one-hot into first 37 columns
    encoded_atoms[:, :N_ATOM_SUBTYPE] = subtype_one_hot

    for i, atom in enumerate(mol.GetAtoms()):
        pos = conf.GetAtomPosition(atom.GetIdx())
        coords[i] = [pos.x, pos.y, pos.z]

        # Offsets after first 37 subtype columns
        base = N_ATOM_SUBTYPE

        # Hybridization: columns 37, 38, 39
        hybridization = atom.GetHybridization()
        if hybridization == Chem.HybridizationType.SP:
            encoded_atoms[i, base + 0] = 1
        elif hybridization == Chem.HybridizationType.SP2:
            encoded_atoms[i, base + 1] = 1
        elif hybridization == Chem.HybridizationType.SP3:
            encoded_atoms[i, base + 2] = 1

        # Formal charge, ring, aromatic: columns 40, 41, 42
        encoded_atoms[i, base + 3] = 1 if atom.GetFormalCharge() != 0 else 0
        encoded_atoms[i, base + 4] = 1 if atom.IsInRing() else 0
        encoded_atoms[i, base + 5] = 1 if atom.GetIsAromatic() else 0

        # Residue grouping: columns 43..51
        pdb_info = atom.GetPDBResidueInfo()
        residue = pdb_info.GetResidueName().strip() if pdb_info is not None else ""

        if residue in ['ASP', 'GLU']:
            encoded_atoms[i, base + 6] = 1
        elif residue in ['LYS', 'ARG']:
            encoded_atoms[i, base + 7] = 1
        elif residue == 'HIS':
            encoded_atoms[i, base + 8] = 1
        elif residue == 'CYS':
            encoded_atoms[i, base + 9] = 1
        elif residue in ['ASN', 'GLN', 'SER', 'THR']:
            encoded_atoms[i, base + 10] = 1
        elif residue == 'GLY':
            encoded_atoms[i, base + 11] = 1
        elif residue == 'PRO':
            encoded_atoms[i, base + 12] = 1
        elif residue in ['PHE', 'TYR', 'TRP']:
            encoded_atoms[i, base + 13] = 1
        elif residue in ['ALA', 'ILE', 'LEU', 'MET', 'VAL']:
            encoded_atoms[i, base + 14] = 1

    return coords.astype(np.float32), encoded_atoms.astype(output_dtype)

def compute_inverse_pairwise_distances(df):
    """
    Compute the pairwise Euclidean distances between residues based on their 3D coordinates.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'X', 'Y', 'Z' coordinates and 'NewIndex' as index.

    Returns:
    pd.DataFrame: A DataFrame containing the pairwise distance matrix.
    """
    # Extract the coordinates (X, Y, Z)
    coordinates = df[['X', 'Y', 'Z']].values

    # Calculate pairwise distances using broadcasting
    diff = coordinates[:, np.newaxis, :] - coordinates[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=-1))

    # Compute inverse distance (1/d)
    with np.errstate(divide='ignore'):  # Ignore division by zero warning
        inverse_distances = 1 / distances

    # Set diagonal elements (self-distances) to 1
    np.fill_diagonal(inverse_distances, 1)

    # Cap values at 1
    inverse_distances = np.minimum(inverse_distances, 1)

    return inverse_distances

def pdb_to_dataframe(pdb_file):
    """
    Load a PDB file using MDAnalysis and convert key atom information to a pandas DataFrame.
    """
    u = mda.Universe(pdb_file)
    
    # Extract atom-related data: atom name, residue name, residue ID, and chain ID
    atom_data = {
        'Atom Name': u.atoms.names,
        'Residue Name': u.atoms.resnames,
        'Residue ID': u.atoms.resids,
        'Chain ID': u.atoms.segids,
        'X': u.atoms.positions[:, 0],
        'Y': u.atoms.positions[:, 1],
        'Z': u.atoms.positions[:, 2],
    }
    
    # Create a pandas DataFrame from the atom data
    df = pd.DataFrame(atom_data)
    
    return df

def compute_inverse_pairwise_distances_from_coords(coords):
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=-1))

    with np.errstate(divide='ignore'):
        inverse_distances = 1.0 / distances

    np.fill_diagonal(inverse_distances, 1.0)
    inverse_distances = np.minimum(inverse_distances, 1.0)

    return inverse_distances

In [56]:
max_atoms = 200
output_dir = "cholesterol-rdkit-5A/positive"
os.makedirs(output_dir, exist_ok=True)

positive_files = glob.glob("filtered-rdkit-distinct-5A/positive/*.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

for file in positive_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    combined_matrix = inverse_distance @ encoded_matrix

    num_atoms = inverse_distance.shape[0]

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

    base_name = os.path.splitext(os.path.basename(file))[0]
    output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
    np.save(output_path, combined_matrix)

    #print(f"Saved: {output_path}")

[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/positive/5NM4-filtered.pdb: pdb_df has 72 rows, RDKit mol has 67 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/positive/5NM4-filtered.pdb


In [57]:
max_atoms = 200
output_dir = "cholesterol-rdkit-5A/unlabeled"
os.makedirs(output_dir, exist_ok=True)

unlabeled_files = glob.glob("filtered-rdkit-distinct-5A/unlabeled/*.pdb")
unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

for file in unlabeled_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    combined_matrix = inverse_distance @ encoded_matrix

    num_atoms = inverse_distance.shape[0]

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

    base_name = os.path.splitext(os.path.basename(file))[0]
    output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
    np.save(output_path, combined_matrix)

    #print(f"Saved: {output_path}")

[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/1LRI-f1.pdb: pdb_df has 84 rows, RDKit mol has 80 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/1LRI-f1.pdb
OXT went to UNKNOWN column
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/4EIY-f1.pdb: pdb_df has 91 rows, RDKit mol has 85 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4EIY-f1.pdb


[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/4XP1-f4.pdb: pdb_df has 74 rows, RDKit mol has 69 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4XP1-f4.pdb
[SKIP] RDKit could not parse: filtered-rdkit-distinct-5A/unlabeled/4XPF-f4.pdb
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4XPF-f4.pdb


[10:42:28] Explicit valence for atom # 98 C, 5, is greater than permitted


[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU7-f1.pdb: pdb_df has 108 rows, RDKit mol has 100 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU7-f1.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU7-f4.pdb: pdb_df has 67 rows, RDKit mol has 64 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU7-f4.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU8-f5.pdb: pdb_df has 110 rows, RDKit mol has 103 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU8-f5.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IUA-f1.pdb: pdb_df has 109 rows, RDKit mol has 101 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IUA-f1.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IUA-f2.pdb: pdb_df has 117 rows, RDKit mol has 103 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A

In [58]:
feature_names = [
    'C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3',
    'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3',

    # Oxygen (8)
    'O', 'OH', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1',

    # Nitrogen (9)
    'N', 'NE', 'NE1', 'NE2', 'ND1', 'ND2', 'NZ', 'NH1', 'NH2',

    # Sulfur (2)
    'SD', 'SG',

    # Unknown (1)
    'UNKNOWN',

    "hybridization_SP",                         
    "hybridization_SP2",                        
    "hybridization_SP3",                        

    "formal_charge_nonzero",                    

    "is_in_ring",                               
    "is_aromatic",                              

    "residue_acidic_ASP_GLU",                   
    "residue_basic_LYS_ARG",                    
    "residue_HIS",                              
    "residue_CYS",                              
    "residue_polar_ASN_GLN_SER_THR",            
    "residue_GLY",                              
    "residue_PRO",                              
    "residue_aromatic_PHE_TYR_TRP",             
    "residue_hydrophobic_ALA_ILE_LEU_MET_VAL"   
]

In [59]:
max_atoms = 200
output_dir = "cholesterol-rdkit-5A/positive"
np.set_printoptions(threshold=np.inf)

positive_files = glob.glob("filtered-rdkit-distinct-5A/positive/*.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

# Trackers
feature_usage = None                 # total atom-level usage
feature_file_presence = None         # per-file presence
num_features = None
total_atoms_all_files = 0
total_files = len(positive_files)

for file in positive_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    # Initialize trackers
    if feature_usage is None:
        num_features = encoded_matrix.shape[1]
        feature_usage = np.zeros(num_features, dtype=np.int64)
        feature_file_presence = np.zeros(num_features, dtype=np.int64)

    num_atoms = inverse_distance.shape[0]
    total_atoms_all_files += num_atoms

    # ---- Atom-level usage ----
    feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)

    # ---- File-level presence ----
    feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

    combined_matrix = inverse_distance @ encoded_matrix

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

# ---- AFTER LOOP ----

print("\n=== FEATURE USAGE SUMMARY ===")
print(f"Total atoms: {total_atoms_all_files}")
print(f"Total files: {total_files}\n")

for i in range(num_features):
    atom_pct = (feature_usage[i] / total_atoms_all_files) * 100 if total_atoms_all_files > 0 else 0
    file_pct = (feature_file_presence[i] / total_files) * 100 if total_files > 0 else 0

    print(
        f"{i:2d} | {feature_names[i]:40s} | "
        f"atoms: {feature_usage[i]:8d} ({atom_pct:7.3f}%) | "
        f"files: {feature_file_presence[i]:4d}/{total_files} ({file_pct:6.2f}%)"
    )

# ---- UNUSED FEATURES ----

unused_features = np.where(feature_usage == 0)[0]

print("\n=== UNUSED FEATURES ===")
for i in unused_features:
    print(f"{i:2d} | {feature_names[i]}")

[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/positive/5NM4-filtered.pdb: pdb_df has 72 rows, RDKit mol has 67 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/positive/5NM4-filtered.pdb

=== FEATURE USAGE SUMMARY ===
Total atoms: 32804
Total files: 770

 0 | C                                        | atoms:     3949 ( 12.038%) | files:  769/770 ( 99.87%)
 1 | CA                                       | atoms:     3949 ( 12.038%) | files:  769/770 ( 99.87%)
 2 | CB                                       | atoms:     3695 ( 11.264%) | files:  769/770 ( 99.87%)
 3 | CD                                       | atoms:      352 (  1.073%) | files:  302/770 ( 39.22%)
 4 | CD1                                      | atoms:     2029 (  6.185%) | files:  729/770 ( 94.68%)
 5 | CD2                                      | atoms:     1528 (  4.658%) | files:  677/770 ( 87.92%)
 6 | CE                                       | atoms:      155 (  0.473%) | files:  131/770 ( 17.01%

In [60]:
max_atoms = 200
output_dir = "cholesterol-rdkit-5A/unlabeled"
np.set_printoptions(threshold=np.inf)

unlabeled_files = glob.glob("filtered-rdkit-distinct-5A/unlabeled/*.pdb")
unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# Trackers
feature_usage = None                 # total atom-level usage
feature_file_presence = None         # per-file presence
num_features = None
total_atoms_all_files = 0
total_files = len(unlabeled_files)

for file in unlabeled_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    # Initialize trackers
    if feature_usage is None:
        num_features = encoded_matrix.shape[1]
        feature_usage = np.zeros(num_features, dtype=np.int64)
        feature_file_presence = np.zeros(num_features, dtype=np.int64)

    num_atoms = inverse_distance.shape[0]
    total_atoms_all_files += num_atoms

    # ---- Atom-level usage ----
    feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)

    # ---- File-level presence ----
    feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

    combined_matrix = inverse_distance @ encoded_matrix

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

# ---- AFTER LOOP ----

print("\n=== FEATURE USAGE SUMMARY ===")
print(f"Total atoms: {total_atoms_all_files}")
print(f"Total files: {total_files}\n")

for i in range(num_features):
    atom_pct = (feature_usage[i] / total_atoms_all_files) * 100 if total_atoms_all_files > 0 else 0
    file_pct = (feature_file_presence[i] / total_files) * 100 if total_files > 0 else 0

    print(
        f"{i:2d} | {feature_names[i]:40s} | "
        f"atoms: {feature_usage[i]:8d} ({atom_pct:7.3f}%) | "
        f"files: {feature_file_presence[i]:4d}/{total_files} ({file_pct:6.2f}%)"
    )

# ---- UNUSED FEATURES ----

unused_features = np.where(feature_usage == 0)[0]

print("\n=== UNUSED FEATURES ===")
for i in unused_features:
    print(f"{i:2d} | {feature_names[i]}")

[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/1LRI-f1.pdb: pdb_df has 84 rows, RDKit mol has 80 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/1LRI-f1.pdb
OXT went to UNKNOWN column
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/4EIY-f1.pdb: pdb_df has 91 rows, RDKit mol has 85 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4EIY-f1.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/4XP1-f4.pdb: pdb_df has 74 rows, RDKit mol has 69 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4XP1-f4.pdb


[10:42:46] Explicit valence for atom # 98 C, 5, is greater than permitted


[SKIP] RDKit could not parse: filtered-rdkit-distinct-5A/unlabeled/4XPF-f4.pdb
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/4XPF-f4.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU7-f1.pdb: pdb_df has 108 rows, RDKit mol has 100 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU7-f1.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU7-f4.pdb: pdb_df has 67 rows, RDKit mol has 64 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU7-f4.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IU8-f5.pdb: pdb_df has 110 rows, RDKit mol has 103 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IU8-f5.pdb
[WARNING] Atom count mismatch for filtered-rdkit-distinct-5A/unlabeled/5IUA-f1.pdb: pdb_df has 109 rows, RDKit mol has 101 atoms
[SKIP] Failed encoding for filtered-rdkit-distinct-5A/unlabeled/5IUA-f1.pdb
[WARNING] Atom count mismat

In [61]:
# #max_atoms = 200
# output_dir = "cholesterol-rdkit-5A/unlabeled"
# np.set_printoptions(threshold=np.inf)

# unlabeled_files = glob.glob("/home/alexhernandez/CholBindNet/GNN/CLR-PDB/*.pdb")
# unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# # Trackers
# feature_usage = None                 # total atom-level usage
# feature_file_presence = None         # per-file presence
# num_features = None
# total_atoms_all_files = 0
# total_files = len(unlabeled_files)

# for file in unlabeled_files:
#     coords, encoded_matrix = rdkit_coords_and_encoding(file)
#     if coords is None:
#         continue
#     inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

#     if inverse_distance.shape[0] != encoded_matrix.shape[0]:
#         raise ValueError(
#             f"Atom mismatch in {file}: "
#             f"inverse_distance={inverse_distance.shape}, "
#             f"encoded_matrix={encoded_matrix.shape}"
#         )

#     # Initialize trackers
#     if feature_usage is None:
#         num_features = encoded_matrix.shape[1]
#         feature_usage = np.zeros(num_features, dtype=np.int64)
#         feature_file_presence = np.zeros(num_features, dtype=np.int64)

#     num_atoms = inverse_distance.shape[0]
#     total_atoms_all_files += num_atoms

#     # ---- Atom-level usage ----
#     feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)

#     # ---- File-level presence ----
#     feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

#     combined_matrix = inverse_distance @ encoded_matrix

#     # if num_atoms > max_atoms:
#     #     print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
#     #     raise Exception("Too many atoms!")

#     # combined_matrix = np.pad(
#     #     combined_matrix,
#     #     ((0, max_atoms - num_atoms), (0, 0)),
#     #     mode='constant'
#     # )

# # ---- AFTER LOOP ----

# print("\n=== FEATURE USAGE SUMMARY ===")
# print(f"Total atoms: {total_atoms_all_files}")
# print(f"Total files: {total_files}\n")

# for i in range(num_features):
#     atom_pct = (feature_usage[i] / total_atoms_all_files) * 100 if total_atoms_all_files > 0 else 0
#     file_pct = (feature_file_presence[i] / total_files) * 100 if total_files > 0 else 0

#     print(
#         f"{i:2d} | {feature_names[i]:40s} | "
#         f"atoms: {feature_usage[i]:8d} ({atom_pct:7.3f}%) | "
#         f"files: {feature_file_presence[i]:4d}/{total_files} ({file_pct:6.2f}%)"
#     )

# # ---- UNUSED FEATURES ----

# unused_features = np.where(feature_usage == 0)[0]

# print("\n=== UNUSED FEATURES ===")
# for i in unused_features:
#     print(f"{i:2d} | {feature_names[i]}")